In [16]:
import pandas as pd
import numpy as np

# Wczytanie gotowych danych z poprzednich etapów
df_fi_hard = pd.read_parquet("dane/interim/fact_inka_hard_records_2023_2026.parquet")  # jeśli masz taki zapis
pelny_kalendarz = pd.read_parquet("dane/interim/kalendarz_pelny_towid.parquet")
slownik_towid = pd.read_parquet("dane/interim/slownik_towid_kategorie.parquet")

In [49]:
kontrola = df_fi_hard[
    ['TypRuchu', 'TypDok', 'Dokument', 'WplywNaStan', 'MetodaLiczenia', 'Mnoznik', 'CzyNiechciane']
].drop_duplicates(subset=['TypRuchu', 'TypDok']).sort_values('TypDok')

print(kontrola.to_string(index=False))

    TypRuchu  TypDok       Dokument  WplywNaStan MetodaLiczenia  Mnoznik  CzyNiechciane
   przyjecie       2             PZ         True             IP        1          False
   neutralny       4           ZWFD        False           Brak        0          False
       zwrot       8          ZWPAR         True             IP        1          False
   przyjecie       9             PW         True             IP        1          False
     rozchod      10             RW         True             IP       -1          False
          bo      14             BO         True       RESET_IP        1          False
    remanent      16            REM         True       RESET_IP        1          False
    przecena      18          PRZEC        False           Brak        0           True
    przecena      19     ZAMR_PRZEC        False           Brak        0           True
    sprzedaz      21             DF         True             IP       -1          False
      strata      23            

In [50]:
# ============================================================
# Obliczenie ruchu ilościowego per wiersz, wg MetodaLiczenia
# ============================================================

def oblicz_ruch(row):
    if row['MetodaLiczenia'] == 'Brak':
        return 0.0
    elif row['MetodaLiczenia'] == 'IP':
        return row['IloscPlus'] * row['Mnoznik']
    elif row['MetodaLiczenia'] == 'RESET_IP':
        return row['IloscPlus']  # wartość do resetu, nie ruch - obsłużymy osobno
    elif row['MetodaLiczenia'] == 'IP_IM_DELTA':
        return (row['IloscPlus'] - row['IloscMinus']) * row['Mnoznik']
    else:
        return 0.0

# Wersja wektoryzowana (szybsza niż .apply na 4mln wierszy)
df_fi_hard['RuchIlosciowy'] = 0.0

maska_ip = df_fi_hard['MetodaLiczenia'] == 'IP'
df_fi_hard.loc[maska_ip, 'RuchIlosciowy'] = (
    df_fi_hard.loc[maska_ip, 'IloscPlus'] * df_fi_hard.loc[maska_ip, 'Mnoznik']
)

maska_delta = df_fi_hard['MetodaLiczenia'] == 'IP_IM_DELTA'
df_fi_hard.loc[maska_delta, 'RuchIlosciowy'] = (
    (df_fi_hard.loc[maska_delta, 'IloscPlus'] - df_fi_hard.loc[maska_delta, 'IloscMinus']) 
    * df_fi_hard.loc[maska_delta, 'Mnoznik']
)

# RESET_IP - osobna kolumna, bo to nie jest ruch tylko wartość do ustawienia
df_fi_hard['WartoscResetu'] = np.where(
    df_fi_hard['MetodaLiczenia'] == 'RESET_IP', df_fi_hard['IloscPlus'], np.nan
)

print(df_fi_hard[['TypRuchu', 'MetodaLiczenia', 'IloscPlus', 'IloscMinus', 'Mnoznik', 'RuchIlosciowy', 'WartoscResetu']].drop_duplicates(subset='TypRuchu'))

             TypRuchu MetodaLiczenia  IloscPlus  IloscMinus  Mnoznik  \
0           neutralny           Brak       1.00         0.0        0   
4            sprzedaz             IP       1.00         0.0       -1   
162            strata             IP       0.27         0.0       -1   
201      przesuniecie    IP_IM_DELTA       0.00         2.0        1   
313          przecena           Brak      55.00         0.0        0   
331          remanent       RESET_IP       0.00        24.0        1   
3816        przyjecie             IP      18.00         0.0        1   
5910          rozchod             IP       1.00         0.0       -1   
9540      rozbieznosc    IP_IM_DELTA       0.00        10.0        1   
12623           zwrot             IP       1.00         0.0        1   
3070413            bo       RESET_IP       0.00         0.0        1   

         RuchIlosciowy  WartoscResetu  
0                 0.00            NaN  
4                -1.00            NaN  
162            

In [20]:
ruch_dzienny = df_fi_hard.groupby(['TowId', 'Data']).agg(
    RuchIlosciowyDzien=('RuchIlosciowy', 'sum'),
    # Jeśli tego dnia był BO/REM, bierzemy ostatnią wartość resetu (zwykle 1 na dzień)
    WartoscResetuDzien=('WartoscResetu', 'max')  
).reset_index()

print(ruch_dzienny.head(10))
print(f"\nDni z resetem (BO/REM): {ruch_dzienny['WartoscResetuDzien'].notna().sum()}")

   TowId       Data  RuchIlosciowyDzien  WartoscResetuDzien
0      5 2023-01-01                 0.0                 0.0
1      5 2023-02-19                 0.0                 0.0
2      5 2024-02-25                 0.0                 0.0
3      5 2024-11-10                 0.0                 0.0
4     15 2023-01-01                 0.0                 3.0
5     15 2023-02-04                -1.0                 NaN
6     15 2023-02-13                -1.0                 NaN
7     15 2023-02-19                 0.0                 0.0
8     15 2023-03-14                 3.0                 NaN
9     15 2023-03-29                -1.0                 NaN

Dni z resetem (BO/REM): 66628


In [21]:
remanenty_przyklad = df_fi_hard[df_fi_hard['TypRuchu']=='remanent'][
    ['TowId', 'Data', 'DokId', 'IloscPlus', 'IloscMinus', 'NazwaTow']
].head(30)

print(remanenty_przyklad.to_string(index=False))

# Czy remanenty MAJĄ zarówno IloscPlus>0 jak i IloscMinus>0 w różnych wierszach?
print("\nRemanenty z IloscPlus>0:", (df_fi_hard[df_fi_hard['TypRuchu']=='remanent']['IloscPlus']>0).sum())
print("Remanenty z IloscMinus>0:", (df_fi_hard[df_fi_hard['TypRuchu']=='remanent']['IloscMinus']>0).sum())

 TowId       Data   DokId  IloscPlus  IloscMinus                                NazwaTow
  6255 2023-01-02 1166267       0.00      24.000            MASŁO POLSKIE 200G MLEKOVITA
 49637 2023-01-02 1166481       0.00      11.771                     POMIDOR MALINOWY KG
 50008 2023-01-04 1168346       1.50       7.131            MANDARYNKA 1KG LUZ HISZPANIA
 49851 2023-01-05 1169011       0.36      -0.462                POMARAŃCZA HISZPANIA 1KG
 49624 2023-01-05 1169011       0.00      -9.665                               BANAN /KG
 49627 2023-01-05 1169011       0.00       6.157                       CYTRYNY TURCJA KG
 49633 2023-01-05 1169011       0.00       0.035                 OGÓREK WĘŻOWY POLSKA KG
 17948 2023-01-09 1171549       0.00     -17.000        DROPSY MENTOS FRUIT 40G PERFETTI
 49840 2023-02-09 1207905       4.00       3.595                 AVOCADO KAL.22/2   1SZT
 49840 2023-02-09 1207930       2.00       4.000                 AVOCADO KAL.22/2   1SZT
 51471 2023-02-09 120

In [22]:
# Test: czy remanent jako "delta" (IloscPlus - IloscMinus) dodana do biezącego stanu
# daje bardziej sensowne wyniki niż "reset do IloscPlus"?

# Sprawdź pełną historię jednego konkretnego, prostego TowId z kilkoma remanentami
przyklad = df_fi_hard[df_fi_hard['TowId']==6255].sort_values('Data')[
    ['Data', 'DokId', 'TypRuchu', 'MetodaLiczenia', 'IloscPlus', 'IloscMinus', 'Mnoznik', 'NazwaTow']
]
print(przyklad.to_string(index=False))

      Data   DokId     TypRuchu MetodaLiczenia  IloscPlus  IloscMinus  Mnoznik                     NazwaTow
2023-01-01 1982216           bo       RESET_IP       24.0         0.0        1 MASŁO POLSKIE 200G MLEKOVITA
2023-01-01 1982218           bo       RESET_IP        0.0         0.0        1 MASŁO POLSKIE 200G MLEKOVITA
2023-01-02 1166267     remanent       RESET_IP        0.0        24.0        1 MASŁO POLSKIE 200G MLEKOVITA
2023-01-03 1167701      rozchod             IP        4.0         0.0       -1 MASŁO POLSKIE 200G MLEKOVITA
2023-01-03 1167351     sprzedaz             IP        1.0         0.0       -1 MASŁO POLSKIE 200G MLEKOVITA
2023-01-03 1167251    przyjecie             IP       20.0         0.0        1 MASŁO POLSKIE 200G MLEKOVITA
2023-01-05 1169189     sprzedaz             IP        3.0         0.0       -1 MASŁO POLSKIE 200G MLEKOVITA
2023-01-05 1169326     sprzedaz             IP        1.0         0.0       -1 MASŁO POLSKIE 200G MLEKOVITA
2023-01-05 1169937     sprze

In [23]:
# Test 1: czy remanenty/BO faktycznie mają rozłączne Plus/Minus (jedno zawsze 0)?
reset_rows = df_fi_hard[df_fi_hard['MetodaLiczenia']=='RESET_IP']
oba_niezerowe = reset_rows[(reset_rows['IloscPlus']>0) & (reset_rows['IloscMinus']>0)]
print(f"Wiersze RESET_IP z OBOMA kolumnami niezerowymi: {len(oba_niezerowe)} z {len(reset_rows)}")

# Test 2: zbuduj WartoscResetu jako sumę i sprawdź na przykładzie masła
df_fi_hard['WartoscResetu_v2'] = np.where(
    df_fi_hard['MetodaLiczenia'] == 'RESET_IP',
    df_fi_hard['IloscPlus'] + df_fi_hard['IloscMinus'],
    np.nan
)

przyklad = df_fi_hard[df_fi_hard['TowId']==6255].sort_values('Data').head(15)[
    ['Data', 'DokId', 'TypRuchu', 'MetodaLiczenia', 'IloscPlus', 'IloscMinus', 'WartoscResetu_v2']
]
print(przyklad.to_string(index=False))

Wiersze RESET_IP z OBOMA kolumnami niezerowymi: 18338 z 87120
      Data   DokId  TypRuchu MetodaLiczenia  IloscPlus  IloscMinus  WartoscResetu_v2
2023-01-01 1982216        bo       RESET_IP       24.0         0.0              24.0
2023-01-01 1982218        bo       RESET_IP        0.0         0.0               0.0
2023-01-02 1166267  remanent       RESET_IP        0.0        24.0              24.0
2023-01-03 1167701   rozchod             IP        4.0         0.0               NaN
2023-01-03 1167351  sprzedaz             IP        1.0         0.0               NaN
2023-01-03 1167251 przyjecie             IP       20.0         0.0               NaN
2023-01-05 1169189  sprzedaz             IP        3.0         0.0               NaN
2023-01-05 1169326  sprzedaz             IP        1.0         0.0               NaN
2023-01-05 1169937  sprzedaz             IP        1.0         0.0               NaN
2023-01-05 1169960  sprzedaz             IP        1.0         0.0               NaN
202

In [24]:
# Przykłady RESET_IP z obiema kolumnami niezerowymi
oba_niezerowe_przyklad = df_fi_hard[
    (df_fi_hard['MetodaLiczenia']=='RESET_IP') & 
    (df_fi_hard['IloscPlus']>0) & (df_fi_hard['IloscMinus']>0)
][['TowId', 'Data', 'DokId', 'TypRuchu', 'IloscPlus', 'IloscMinus', 'NazwaTow']].head(20)

print(oba_niezerowe_przyklad.to_string(index=False))

 TowId       Data   DokId TypRuchu  IloscPlus  IloscMinus                                          NazwaTow
 50008 2023-01-04 1168346 remanent        1.5       7.131                      MANDARYNKA 1KG LUZ HISZPANIA
 49840 2023-02-09 1207905 remanent        4.0       3.595                           AVOCADO KAL.22/2   1SZT
 49840 2023-02-09 1207930 remanent        2.0       4.000                           AVOCADO KAL.22/2   1SZT
 49851 2023-02-09 1207930 remanent        1.5       3.974                          POMARAŃCZA HISZPANIA 1KG
 49630 2023-02-09 1207930 remanent        2.5       3.122                           KAPUSTA BIAŁA POLSKA KG
 53719 2023-02-09 1207930 remanent       50.0      79.241                                      ZIEMNIAKI kg
 39201 2023-02-19 1216668 remanent        3.0       7.000                    NAPÓJ GAZ SCHWEPPES TONIC 1,4L
 54770 2023-02-19 1216668 remanent        8.0      12.000                NAPÓJ GAZ SCHWEPPES RUSSCHIAN 1,4L
 55205 2023-02-19 1216668 re

In [25]:
przyklad_towid_mieszany = oba_niezerowe_przyklad['TowId'].iloc[0]

historia = df_fi_hard[df_fi_hard['TowId']==przyklad_towid_mieszany].sort_values('Data')[
    ['Data', 'DokId', 'TypRuchu', 'MetodaLiczenia', 'IloscPlus', 'IloscMinus', 'Mnoznik']
]
print(f"TowId: {przyklad_towid_mieszany}")
print(historia.head(25).to_string(index=False))

TowId: 50008
      Data   DokId  TypRuchu MetodaLiczenia  IloscPlus  IloscMinus  Mnoznik
2023-01-01 1982216        bo       RESET_IP      9.771       0.000        1
2023-01-01 1982218        bo       RESET_IP      0.000       0.000        1
2023-01-02 1166391  sprzedaz             IP      0.378       0.000       -1
2023-01-02 1166244    strata             IP      0.080       0.000       -1
2023-01-02 1166288  sprzedaz             IP      0.734       0.000       -1
2023-01-02 1166247  sprzedaz             IP      0.394       0.000       -1
2023-01-03 1167165  sprzedaz             IP      0.180       0.000       -1
2023-01-03 1167334  sprzedaz             IP      0.368       0.000       -1
2023-01-03 1167467  sprzedaz             IP      0.346       0.000       -1
2023-01-04 1168340    strata             IP      0.160       0.000       -1
2023-01-04 1168346  remanent       RESET_IP      1.500       7.131        1
2023-01-04 1168377 neutralny           Brak     30.000       0.000        0

In [26]:
# WALIDACJA: dla każdego RESET_IP, IloscMinus powinno zgadzać się 
# z naszym wyliczonym stanem księgowym TUŻ PRZED tym wierszem

df_fi_hard['WartoscResetu'] = np.where(
    df_fi_hard['MetodaLiczenia'] == 'RESET_IP',
    df_fi_hard['IloscPlus'],
    np.nan
)

# Policzmy stan "księgowy" (bez uwzględniania resetów) narastająco, tylko do porównania
df_fi_hard_sorted = df_fi_hard.sort_values(['TowId', 'Data', 'DokId'])
df_fi_hard_sorted['StanKsiegowyNarastajaco'] = (
    df_fi_hard_sorted.groupby('TowId')['RuchIlosciowy'].cumsum()
)

# Dla wierszy RESET_IP: stan księgowy TUŻ PRZED (czyli narastająco minus bieżący wiersz, 
# bo RuchIlosciowy dla RESET_IP i tak wynosi 0 w naszej metodzie)
maska_reset = df_fi_hard_sorted['MetodaLiczenia'] == 'RESET_IP'
test = df_fi_hard_sorted[maska_reset][
    ['TowId','Data','DokId','IloscMinus','StanKsiegowyNarastajaco']
].copy()
test['Roznica'] = (test['StanKsiegowyNarastajaco'] - test['IloscMinus']).round(3)

print(f"Zgodność (różnica ~0): {(test['Roznica'].abs() < 0.01).mean()*100:.1f}%")
print(test[test['Roznica'].abs() >= 0.01].head(10))

Zgodność (różnica ~0): 76.5%
         TowId       Data    DokId  IloscMinus  StanKsiegowyNarastajaco  \
183420      15 2023-02-19  1216669         1.0                     -2.0   
1501868     15 2024-02-25  1601036         2.0                      0.0   
2501740     15 2024-11-10  1842411        -5.0                     -7.0   
2812848     15 2025-02-03  1911588        -3.0                    -10.0   
173423      49 2023-02-19  1216669         5.0                      0.0   
1500263     49 2024-02-25  1601036         4.0                     -1.0   
2491308     49 2024-11-10  1842411         4.0                     -1.0   
173421      53 2023-02-19  1216669         3.0                     -2.0   
1500269     53 2024-02-25  1601036         4.0                     -1.0   
2491314     53 2024-11-10  1842411         2.0                     -3.0   

         Roznica  
183420      -3.0  
1501868     -2.0  
2501740     -2.0  
2812848     -7.0  
173423      -5.0  
1500263     -5.0  
2491308     

In [30]:
# ============================================================
# Poprawny test: stan z UWZGLĘDNIENIEM resetów, sprawdzany na KOLEJNYM resecie
# ============================================================

df_fi_hard_sorted = df_fi_hard.sort_values(['TowId', 'Data', 'DokId']).reset_index(drop=True)

def stan_z_resetem(grupa):
    grupa = grupa.reset_index(drop=True)
    stan = 0.0
    wynik = np.empty(len(grupa))
    for i in range(len(grupa)):
        if grupa.loc[i, 'MetodaLiczenia'] == 'RESET_IP':
            # Zanim zresetujemy - zapisz stan PRZED resetem (do walidacji)
            wynik[i] = stan  # stan tuż przed tym wierszem
            stan = grupa.loc[i, 'IloscPlus']  # reset do nowej wartości fizycznej
        else:
            stan += grupa.loc[i, 'RuchIlosciowy']
            wynik[i] = stan
    grupa['StanPoTymWierszu'] = wynik
    return grupa

# Test na próbce kilku TowId z błędami z poprzedniego testu (15, 49, 53)
test_towid = [15, 49, 53]
test_dane_wejscie = df_fi_hard_sorted[df_fi_hard_sorted['TowId'].isin(test_towid)].copy()

# Bezpieczna metoda - jawnie przywracamy TowId po grupowaniu
wyniki = []
for towid, grupa in test_dane_wejscie.groupby('TowId'):
    grupa_wynik = stan_z_resetem(grupa)
    grupa_wynik['TowId'] = towid
    wyniki.append(grupa_wynik)

test_dane = pd.concat(wyniki, ignore_index=True)

# Sprawdźmy zgodność: dla wierszy RESET_IP, 'StanPoTymWierszu' (czyli stan TUŻ PRZED resetem) 
# powinien zgadzać się z IloscMinus
maska_reset = test_dane['MetodaLiczenia'] == 'RESET_IP'
test_dane.loc[maska_reset, 'Roznica'] = (
    test_dane.loc[maska_reset, 'StanPoTymWierszu'] - test_dane.loc[maska_reset, 'IloscMinus']
).round(3)

print(test_dane[maska_reset][['TowId','Data','DokId','IloscMinus','StanPoTymWierszu','Roznica']])

     TowId       Data    DokId  IloscMinus  StanPoTymWierszu  Roznica
0       15 2023-01-01  1982216         0.0               0.0      0.0
1       15 2023-01-01  1982218         0.0               3.0      3.0
4       15 2023-02-19  1216669         1.0              -2.0     -3.0
31      15 2024-02-25  1601036         2.0               2.0      0.0
39      15 2024-11-10  1842411        -5.0              -5.0      0.0
49      15 2025-02-03  1911588        -3.0              -3.0      0.0
73      49 2023-01-01  1982216         0.0               0.0      0.0
74      49 2023-01-01  1982218         0.0               5.0      5.0
75      49 2023-02-19  1216669         5.0               0.0     -5.0
93      49 2024-02-25  1601036         4.0               4.0      0.0
109     49 2024-11-10  1842411         4.0               4.0      0.0
123     53 2023-01-01  1982216         0.0               0.0      0.0
124     53 2023-01-01  1982218         0.0               5.0      5.0
133     53 2023-02-1

In [32]:
bo_wszystkie = df_fi_hard[
    (df_fi_hard['TowId']==15) & (df_fi_hard['TypRuchu']=='bo')
][['DokId', 'Kolejnosc', 'NrPozycji', 'TowId', 'IloscPlus', 'IloscMinus', 'Data', 'NrDok']]

print(bo_wszystkie.to_string(index=False))

  DokId  Kolejnosc  NrPozycji  TowId  IloscPlus  IloscMinus       Data   NrDok
1982216          2          2     15        3.0         0.0 2023-01-01 BO/23/1
1982218          2          2     15        0.0         0.0 2023-01-01 BO/23/2


In [33]:
# ============================================================
# POPRAWIONA logika: sumuj RESET_IP per (TowId, Data), nie per pojedynczy wiersz
# ============================================================

df_fi_hard_sorted = df_fi_hard.sort_values(['TowId', 'Data', 'DokId']).reset_index(drop=True)

# Krok 1: dla wierszy RESET_IP, policz sumę IloscPlus per (TowId, Data) - łączy wszystkie dokumenty tego dnia
suma_resetu_dzien = df_fi_hard_sorted[df_fi_hard_sorted['MetodaLiczenia']=='RESET_IP'].groupby(
    ['TowId', 'Data']
)['IloscPlus'].sum().reset_index()
suma_resetu_dzien.columns = ['TowId', 'Data', 'SumaResetuDzien']

# Krok 2: dołącz tę sumę do głównej tabeli
df_fi_hard_sorted = df_fi_hard_sorted.merge(suma_resetu_dzien, on=['TowId','Data'], how='left')

def stan_z_resetem_v2(grupa):
    grupa = grupa.reset_index(drop=True)
    stan = 0.0
    wynik = np.empty(len(grupa))
    for i in range(len(grupa)):
        if grupa.loc[i, 'MetodaLiczenia'] == 'RESET_IP':
            wynik[i] = stan  # stan przed resetem (walidacja)
            stan = grupa.loc[i, 'SumaResetuDzien']  # reset do SUMY z całego dnia, nie pojedynczego wiersza
        else:
            stan += grupa.loc[i, 'RuchIlosciowy']
            wynik[i] = stan
    grupa['StanPoTymWierszu'] = wynik
    return grupa

test_towid = [15, 49, 53]
test_dane_wejscie = df_fi_hard_sorted[df_fi_hard_sorted['TowId'].isin(test_towid)].copy()

wyniki = []
for towid, grupa in test_dane_wejscie.groupby('TowId'):
    grupa_wynik = stan_z_resetem_v2(grupa)
    grupa_wynik['TowId'] = towid
    wyniki.append(grupa_wynik)

test_dane = pd.concat(wyniki, ignore_index=True)

maska_reset = test_dane['MetodaLiczenia'] == 'RESET_IP'
test_dane.loc[maska_reset, 'Roznica'] = (
    test_dane.loc[maska_reset, 'StanPoTymWierszu'] - test_dane.loc[maska_reset, 'IloscMinus']
).round(3)

print(test_dane[maska_reset][['TowId','Data','DokId','IloscMinus','SumaResetuDzien','StanPoTymWierszu','Roznica']])

     TowId       Data    DokId  IloscMinus  SumaResetuDzien  StanPoTymWierszu  \
0       15 2023-01-01  1982216         0.0              3.0               0.0   
1       15 2023-01-01  1982218         0.0              3.0               3.0   
4       15 2023-02-19  1216669         1.0              0.0               1.0   
31      15 2024-02-25  1601036         2.0              2.0               2.0   
39      15 2024-11-10  1842411        -5.0              0.0              -5.0   
49      15 2025-02-03  1911588        -3.0              1.0              -3.0   
73      49 2023-01-01  1982216         0.0              5.0               0.0   
74      49 2023-01-01  1982218         0.0              5.0               5.0   
75      49 2023-02-19  1216669         5.0              5.0               5.0   
93      49 2024-02-25  1601036         4.0              4.0               4.0   
109     49 2024-11-10  1842411         4.0              4.0               4.0   
123     53 2023-01-01  19822

In [34]:
# ============================================================
# FINALNE liczenie stanu magazynowego — pełny zbiór
# ============================================================

wyniki_pelne = []
for towid, grupa in df_fi_hard_sorted.groupby('TowId'):
    grupa_wynik = stan_z_resetem_v2(grupa)
    grupa_wynik['TowId'] = towid
    wyniki_pelne.append(grupa_wynik)

df_fi_hard_ze_stanem = pd.concat(wyniki_pelne, ignore_index=True)

print(f"Rekordów: {len(df_fi_hard_ze_stanem):,}")

Rekordów: 4,106,901


In [35]:
maska_reset_pelny = df_fi_hard_ze_stanem['MetodaLiczenia'] == 'RESET_IP'

# Wykluczamy pierwszy dzień (BO) z walidacji - tam Roznica z natury nie musi być 0 (patrz wyjaśnienie wyżej)
df_fi_hard_ze_stanem.loc[maska_reset_pelny, 'Roznica'] = (
    df_fi_hard_ze_stanem.loc[maska_reset_pelny, 'StanPoTymWierszu'] - 
    df_fi_hard_ze_stanem.loc[maska_reset_pelny, 'IloscMinus']
).round(3)

walidacja = df_fi_hard_ze_stanem[maska_reset_pelny & (df_fi_hard_ze_stanem['TypRuchu']=='remanent')]

zgodnosc_pct = (walidacja['Roznica'].abs() < 0.01).mean() * 100
print(f"Zgodność dla prawdziwych remanentów (nie BO): {zgodnosc_pct:.2f}%")
print(f"Liczba remanentów sprawdzonych: {len(walidacja):,}")
print(f"\nPrzykłady niezgodności (jeśli są):")
print(walidacja[walidacja['Roznica'].abs() >= 0.01][['TowId','Data','DokId','IloscMinus','StanPoTymWierszu','Roznica']].head(20))

Zgodność dla prawdziwych remanentów (nie BO): 86.93%
Liczba remanentów sprawdzonych: 51,025

Przykłady niezgodności (jeśli są):
       TowId       Data    DokId  IloscMinus  StanPoTymWierszu  Roznica
372       56 2024-11-10  1842411         1.0               4.0      3.0
720       57 2024-11-10  1842411         2.0               8.0      6.0
1255      60 2024-11-10  1842411         4.0               7.0      3.0
2018      65 2023-02-19  1216669        53.3              93.3     40.0
4687      65 2024-02-25  1601036        68.0             192.0    124.0
6161      65 2024-11-10  1842411        21.9              17.9     -4.0
10108     68 2023-02-19  1216669      1942.6            2044.6    102.0
14344     68 2024-02-25  1601036       756.0            1004.5    248.5
16995     68 2024-11-10  1842411       518.2             516.7     -1.5
22459     70 2023-02-19  1216669        16.0              20.0      4.0
22887     70 2024-02-25  1601036         9.5              44.5     35.0
23614   

In [36]:
sprawdz_duplikaty = df_fi_hard[
    (df_fi_hard['TowId']==56) & (df_fi_hard['DokId']==1842411)
][['DokId', 'Kolejnosc', 'NrPozycji', 'TowId', 'IloscPlus', 'IloscMinus', 'Data']]

print(sprawdz_duplikaty.to_string(index=False))

  DokId  Kolejnosc  NrPozycji  TowId  IloscPlus  IloscMinus       Data
1842411       6938       5469     56        1.0         1.0 2024-11-10


In [37]:
historia_56 = df_fi_hard_ze_stanem[df_fi_hard_ze_stanem['TowId']==56].sort_values(['Data','DokId'])[
    ['Data','DokId','TypRuchu','MetodaLiczenia','IloscPlus','IloscMinus','RuchIlosciowy','StanPoTymWierszu']
]
print(historia_56.to_string(index=False))

      Data   DokId  TypRuchu MetodaLiczenia  IloscPlus  IloscMinus  RuchIlosciowy  StanPoTymWierszu
2023-01-01 1982216        bo       RESET_IP        5.0         0.0            0.0               0.0
2023-01-01 1982218        bo       RESET_IP        0.0         0.0            0.0               5.0
2023-01-31 1200558  sprzedaz             IP        1.0         0.0           -1.0               4.0
2023-02-16 1213702  sprzedaz             IP        1.0         0.0           -1.0               3.0
2023-02-19 1216669  remanent       RESET_IP        3.0         3.0            0.0               3.0
2023-03-15 1237728  sprzedaz             IP        1.0         0.0           -1.0               2.0
2023-03-16 1238142 neutralny           Brak        3.0         0.0            0.0               2.0
2023-03-17 1238924 przyjecie             IP        3.0         0.0            3.0               5.0
2023-03-17 1238925  przecena           Brak        2.0         0.0            0.0               5.0


In [38]:
liczba_resetow_dziennie = df_fi_hard[df_fi_hard['MetodaLiczenia']=='RESET_IP'].groupby(
    ['TowId','Data']
).size().reset_index(name='LiczbaResetow')

wiele_resetow = liczba_resetow_dziennie[liczba_resetow_dziennie['LiczbaResetow']>1]
print(f"Dni z więcej niż 1 RESET_IP dla tego samego TowId: {len(wiele_resetow)}")
print(wiele_resetow[wiele_resetow['Data'] != '2023-01-01'].head(20))

Dni z więcej niż 1 RESET_IP dla tego samego TowId: 20241
     TowId       Data  LiczbaResetow
265    252 2023-02-19              2
282    270 2023-02-19              2
286    271 2023-02-19              2
298    275 2023-02-19              2
302    277 2023-02-19              2
306    281 2023-02-19              3
310    282 2023-02-19              2
314    283 2023-02-19              2
321    285 2023-02-19              3
325    286 2023-02-19              2
333    289 2023-02-19              2
337    292 2023-02-19              2
345    298 2023-02-19              2
349    300 2023-02-19              2
361    305 2023-02-19              2
377    315 2023-02-19              2
498    482 2023-02-19              3
502    483 2023-02-19              2
506    486 2023-02-19              2
530    542 2023-02-19              2


In [39]:
sprawdz = df_fi_hard[
    (df_fi_hard['TowId']==252) & (df_fi_hard['Data']=='2023-02-19')
][['DokId','Kolejnosc','TypRuchu','MetodaLiczenia','IloscPlus','IloscMinus']]

print(sprawdz.to_string(index=False))

  DokId  Kolejnosc TypRuchu MetodaLiczenia  IloscPlus  IloscMinus
1216668        653 remanent       RESET_IP       18.0        25.0
1216669       4086 remanent       RESET_IP       18.0        25.0


In [40]:
# Sprawdźmy najpierw czy duplikaty w ramach dnia są ZAWSZE identyczne (to by potwierdziło że to czysty duplikat)
duplikaty_test = df_fi_hard[df_fi_hard['MetodaLiczenia']=='RESET_IP'].groupby(
    ['TowId','Data']
)['IloscPlus'].nunique().reset_index()
duplikaty_test.columns = ['TowId','Data','LiczbaUnikalnychWartosci']

wiele_resetow_z_liczba = wiele_resetow.merge(duplikaty_test, on=['TowId','Data'], how='left')
print(wiele_resetow_z_liczba['LiczbaUnikalnychWartosci'].value_counts())

LiczbaUnikalnychWartosci
1    13916
2     6116
3      198
4       11
Name: count, dtype: int64


In [41]:
# ============================================================
# FINALNA POPRAWKA: deduplikacja identycznych wartości + suma różnych
# ============================================================

suma_resetu_dzien = df_fi_hard_sorted[df_fi_hard_sorted['MetodaLiczenia']=='RESET_IP'].drop_duplicates(
    subset=['TowId', 'Data', 'IloscPlus']
).groupby(['TowId', 'Data'])['IloscPlus'].sum().reset_index()
suma_resetu_dzien.columns = ['TowId', 'Data', 'SumaResetuDzien']

# Podmieniamy starą kolumnę (jeśli była już zmergowana wcześniej)
df_fi_hard_sorted = df_fi_hard_sorted.drop(columns=['SumaResetuDzien'], errors='ignore')
df_fi_hard_sorted = df_fi_hard_sorted.merge(suma_resetu_dzien, on=['TowId','Data'], how='left')

# Test na próbce znanych wcześniej problematycznych TowId
test_towid = [56, 57, 60, 65, 68, 70, 71, 72, 73, 74, 75]
test_dane_wejscie = df_fi_hard_sorted[df_fi_hard_sorted['TowId'].isin(test_towid)].copy()

wyniki = []
for towid, grupa in test_dane_wejscie.groupby('TowId'):
    grupa_wynik = stan_z_resetem_v2(grupa)
    grupa_wynik['TowId'] = towid
    wyniki.append(grupa_wynik)

test_dane = pd.concat(wyniki, ignore_index=True)

maska_reset = (test_dane['MetodaLiczenia'] == 'RESET_IP') & (test_dane['TypRuchu']=='remanent')
test_dane.loc[maska_reset, 'Roznica'] = (
    test_dane.loc[maska_reset, 'StanPoTymWierszu'] - test_dane.loc[maska_reset, 'IloscMinus']
).round(3)

print(test_dane[maska_reset][['TowId','Data','DokId','IloscMinus','SumaResetuDzien','StanPoTymWierszu','Roznica']])

       TowId       Data    DokId  IloscMinus  SumaResetuDzien  \
4         56 2023-02-19  1216669         3.0              3.0   
31        56 2024-02-25  1601036         2.0              2.0   
49        56 2024-11-10  1842411         1.0              1.0   
88        57 2023-02-19  1216669         6.0              6.0   
297       57 2024-02-25  1601036         5.0              5.0   
397       57 2024-11-10  1842411         2.0              2.0   
541       60 2023-02-19  1216669         3.0              3.0   
566       60 2024-02-25  1601036         4.0              5.0   
585       60 2024-11-10  1842411         4.0              4.0   
1062      65 2023-02-19  1216669        53.3              0.0   
3731      65 2024-02-25  1601036        68.0              0.0   
5205      65 2024-11-10  1842411        21.9              0.0   
9147      68 2023-02-19  1216669      1942.6              0.0   
13383     68 2024-02-25  1601036       756.0              0.0   
16034     68 2024-11-10  

In [42]:
# Diagnoza: czy df_fi_hard_sorted ma już starą kolumnę SumaResetuDzien z poprzedniego przebiegu?
print([c for c in df_fi_hard_sorted.columns if 'Suma' in c or 'Reset' in c])

# Sprawdźmy bezpośrednio dla TowId=65 czy merge w ogóle znalazł dopasowanie
print(suma_resetu_dzien[suma_resetu_dzien['TowId']==65])

['WartoscResetu', 'WartoscResetu_v2', 'SumaResetuDzien']
    TowId       Data  SumaResetuDzien
47     65 2023-01-01             92.8
48     65 2023-02-19              0.0
49     65 2024-02-25              0.0
50     65 2024-11-10              0.0


In [43]:
surowe_65 = df_fi_hard[
    (df_fi_hard['TowId']==65) & 
    (df_fi_hard['Data']=='2023-02-19') & 
    (df_fi_hard['MetodaLiczenia']=='RESET_IP')
][['DokId','Kolejnosc','IloscPlus','IloscMinus']]

print(surowe_65.to_string(index=False))

  DokId  Kolejnosc  IloscPlus  IloscMinus
1216669       6713        0.0        53.3


In [44]:
# ============================================================
# UNIWERSALNA REGUŁA: weź niezerową kolumnę; jeśli obie niezerowe, weź IloscPlus
# ============================================================

df_fi_hard_sorted['WartoscFizycznaReset'] = np.where(
    df_fi_hard_sorted['MetodaLiczenia'] == 'RESET_IP',
    np.where(
        df_fi_hard_sorted['IloscPlus'] > 0,
        df_fi_hard_sorted['IloscPlus'],
        df_fi_hard_sorted['IloscMinus']
    ),
    np.nan
)

# Deduplikacja + suma różnych wartości fizycznych per dzień
suma_resetu_dzien = df_fi_hard_sorted[df_fi_hard_sorted['MetodaLiczenia']=='RESET_IP'].drop_duplicates(
    subset=['TowId', 'Data', 'WartoscFizycznaReset']
).groupby(['TowId', 'Data'])['WartoscFizycznaReset'].sum().reset_index()
suma_resetu_dzien.columns = ['TowId', 'Data', 'SumaResetuDzien']

df_fi_hard_sorted = df_fi_hard_sorted.drop(columns=['SumaResetuDzien'], errors='ignore').merge(
    suma_resetu_dzien, on=['TowId','Data'], how='left'
)

# Test na tym konkretnym, wcześniej problematycznym przypadku
print(df_fi_hard_sorted[
    (df_fi_hard_sorted['TowId']==65) & (df_fi_hard_sorted['MetodaLiczenia']=='RESET_IP')
][['Data','DokId','IloscPlus','IloscMinus','WartoscFizycznaReset','SumaResetuDzien']])

           Data    DokId  IloscPlus  IloscMinus  WartoscFizycznaReset  \
1587 2023-01-01  1982216       92.8         0.0                  92.8   
1588 2023-01-01  1982218        0.0         0.0                   0.0   
2018 2023-02-19  1216669        0.0        53.3                  53.3   
4687 2024-02-25  1601036        0.0        68.0                  68.0   
6161 2024-11-10  1842411        0.0        21.9                  21.9   

      SumaResetuDzien  
1587             92.8  
1588             92.8  
2018             53.3  
4687             68.0  
6161             21.9  


In [45]:
def stan_z_resetem_v3(grupa):
    grupa = grupa.reset_index(drop=True)
    stan = 0.0
    wynik = np.empty(len(grupa))
    for i in range(len(grupa)):
        if grupa.loc[i, 'MetodaLiczenia'] == 'RESET_IP':
            wynik[i] = stan
            stan = grupa.loc[i, 'SumaResetuDzien']
        else:
            stan += grupa.loc[i, 'RuchIlosciowy']
            wynik[i] = stan
    grupa['StanPoTymWierszu'] = wynik
    return grupa

test_towid = [56, 57, 60, 65, 68, 70, 71, 72, 73, 74, 75]
test_dane_wejscie = df_fi_hard_sorted[df_fi_hard_sorted['TowId'].isin(test_towid)].copy()

wyniki = []
for towid, grupa in test_dane_wejscie.groupby('TowId'):
    grupa_wynik = stan_z_resetem_v3(grupa)
    grupa_wynik['TowId'] = towid
    wyniki.append(grupa_wynik)

test_dane = pd.concat(wyniki, ignore_index=True)

maska_reset = (test_dane['MetodaLiczenia'] == 'RESET_IP') & (test_dane['TypRuchu']=='remanent')
test_dane.loc[maska_reset, 'Roznica'] = (
    test_dane.loc[maska_reset, 'StanPoTymWierszu'] - test_dane.loc[maska_reset, 'IloscMinus']
).round(3)

print(test_dane[maska_reset][['TowId','Data','IloscMinus','SumaResetuDzien','StanPoTymWierszu','Roznica']])

       TowId       Data  IloscMinus  SumaResetuDzien  StanPoTymWierszu  \
4         56 2023-02-19         3.0              3.0               3.0   
31        56 2024-02-25         2.0              2.0               2.0   
49        56 2024-11-10         1.0              1.0               4.0   
88        57 2023-02-19         6.0              6.0               6.0   
297       57 2024-02-25         5.0              5.0               5.0   
397       57 2024-11-10         2.0              2.0               8.0   
541       60 2023-02-19         3.0              3.0               3.0   
566       60 2024-02-25         4.0              5.0               4.0   
585       60 2024-11-10         4.0              4.0               7.0   
1062      65 2023-02-19        53.3             53.3              93.3   
3731      65 2024-02-25        68.0             68.0             245.3   
5205      65 2024-11-10        21.9             21.9              85.9   
9147      68 2023-02-19      1942.6   

In [46]:
wyniki_pelne = []
for towid, grupa in df_fi_hard_sorted.groupby('TowId'):
    grupa_wynik = stan_z_resetem_v3(grupa)
    grupa_wynik['TowId'] = towid
    wyniki_pelne.append(grupa_wynik)

df_fi_hard_ze_stanem = pd.concat(wyniki_pelne, ignore_index=True)

maska_reset_pelny = (df_fi_hard_ze_stanem['MetodaLiczenia'] == 'RESET_IP') & (df_fi_hard_ze_stanem['TypRuchu']=='remanent')
df_fi_hard_ze_stanem.loc[maska_reset_pelny, 'Roznica'] = (
    df_fi_hard_ze_stanem.loc[maska_reset_pelny, 'StanPoTymWierszu'] - 
    df_fi_hard_ze_stanem.loc[maska_reset_pelny, 'IloscMinus']
).round(3)

walidacja = df_fi_hard_ze_stanem[maska_reset_pelny]
zgodnosc_pct = (walidacja['Roznica'].abs() < 0.01).mean() * 100
print(f"Zgodność: {zgodnosc_pct:.2f}% z {len(walidacja):,} remanentów")

# Rozkład wielkości rozbieżności - żeby zobaczyć czy to małe "szumy" czy duże błędy
print(walidacja['Roznica'].abs().describe())
print(walidacja['Roznica'].abs().quantile([0.5, 0.75, 0.9, 0.95, 0.99]))

Zgodność: 87.54% z 51,025 remanentów
count    51025.000000
mean         1.308244
std         14.056655
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max       2191.100000
Name: Roznica, dtype: float64
0.50     0.0
0.75     0.0
0.90     1.0
0.95     6.0
0.99    24.0
Name: Roznica, dtype: float64


In [47]:
#Flaga jakości — oznacz TowId z dużymi rozbieżnościami, żebyś wiedział które stany traktować z ograniczonym zaufaniem:
duze_rozbieznosci = walidacja[walidacja['Roznica'].abs() > 10]['TowId'].unique()
print(f"TowId z rozbieżnością >10 przy jakimkolwiek remanencie: {len(duze_rozbieznosci)}")

df_fi_hard_ze_stanem['StanMniejPewny'] = df_fi_hard_ze_stanem['TowId'].isin(duze_rozbieznosci)

TowId z rozbieżnością >10 przy jakimkolwiek remanencie: 883


In [48]:
df_fi_hard_ze_stanem.to_parquet(
    "dane/interim/fact_inka_ze_stanem_magazynowym.parquet",
    compression='zstd',
    index=False
)
print(f"Zapisano: {df_fi_hard_ze_stanem.shape}")

Zapisano: (4106901, 40)
